In [20]:
import pandas as pd
import re
import string

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Download required NLTK data (safe to re-run)
for resource in ["stopwords", "punkt", "punkt_tab", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)

import warnings
warnings.filterwarnings('ignore')

In [21]:
df = pd.read_csv('sentiment_dataset.csv')
df.head()

,review,sentiment
0,Would not recommend this item. Review #112,Negative
1,"Five stars, completely satisfied. Review #74",Positive
2,Customer service was unhelpful. Review #125,Negative
3,Good value for money. Review #156,Positive
4,Excellent quality and fast delivery. Review #105,Positive


In [22]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\atiffarooq\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\atiffarooq\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\atiffarooq\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\atiffarooq\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [23]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text
df['clean_review'] = df['review'].apply(clean_text)
df[['review', 'clean_review']]


,review,clean_review
0,Would not recommend this item. Review #112,would not recommend this item review
1,"Five stars, completely satisfied. Review #74",five stars completely satisfied review
2,Customer service was unhelpful. Review #125,customer service was unhelpful review
3,Good value for money. Review #156,good value for money review
4,Excellent quality and fast delivery. Review #105,excellent quality and fast delivery review
...,...,...
495,Good value for money. Review #107,good value for money review
496,Bad experience overall. Review #21,bad experience overall review
497,Very disappointed with the purchase. Review #99,very disappointed with the purchase review
498,The product stopped working quickly. Review #186,the product stopped working quickly review


In [ ]:
df['tokens'] = df['clean_review'].apply(word_tokenize)
df[['clean_review', 'tokens']]

,clean_review,tokens
0,would not recommend this item review,"[would, not, recommend, this, item, review]"
1,five stars completely satisfied review,"[five, stars, completely, satisfied, review]"
2,customer service was unhelpful review,"[customer, service, was, unhelpful, review]"
3,good value for money review,"[good, value, for, money, review]"
4,excellent quality and fast delivery review,"[excellent, quality, and, fast, delivery, review]"
...,...,...
495,good value for money review,"[good, value, for, money, review]"
496,bad experience overall review,"[bad, experience, overall, review]"
497,very disappointed with the purchase review,"[very, disappointed, with, the, purchase, review]"
498,the product stopped working quickly review,"[the, product, stopped, working, quickly, review]"


In [30]:
stop_words = set(stopwords.words('English'))
stop_words

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [ ]:
def remove_stopwords(tokens):
        filtered_words = []
        for word in tokens:
            if word not in stop_words:
                filtered_words.append(word)
        return filtered_words
df['without_stopwords'] = df['tokens'].apply(remove_stopwords)
df[['tokens', 'without_stopwords']]



,tokens,without_stopwords
0,"[would, not, recommend, this, item, review]","[would, recommend, item, review]"
1,"[five, stars, completely, satisfied, review]","[five, stars, completely, satisfied, review]"
2,"[customer, service, was, unhelpful, review]","[customer, service, unhelpful, review]"
3,"[good, value, for, money, review]","[good, value, money, review]"
4,"[excellent, quality, and, fast, delivery, review]","[excellent, quality, fast, delivery, review]"
...,...,...
495,"[good, value, for, money, review]","[good, value, money, review]"
496,"[bad, experience, overall, review]","[bad, experience, overall, review]"
497,"[very, disappointed, with, the, purchase, review]","[disappointed, purchase, review]"
498,"[the, product, stopped, working, quickly, review]","[product, stopped, working, quickly, review]"


In [37]:
def stemming(words):
    stemmed_words = []

    for word in words:
        stemmed_words.append(PorterStemmer().stem(word))

    return stemmed_words

df['stemmed_tokens'] = df['without_stopwords'].apply(stemming)
df[['without_stopwords', 'stemmed_tokens']]

,without_stopwords,stemmed_tokens
0,"[would, recommend, item, review]","[would, recommend, item, review]"
1,"[five, stars, completely, satisfied, review]","[five, star, complet, satisfi, review]"
2,"[customer, service, unhelpful, review]","[custom, servic, unhelp, review]"
3,"[good, value, money, review]","[good, valu, money, review]"
4,"[excellent, quality, fast, delivery, review]","[excel, qualiti, fast, deliveri, review]"
...,...,...
495,"[good, value, money, review]","[good, valu, money, review]"
496,"[bad, experience, overall, review]","[bad, experi, overal, review]"
497,"[disappointed, purchase, review]","[disappoint, purchas, review]"
498,"[product, stopped, working, quickly, review]","[product, stop, work, quickli, review]"


In [38]:
Lemmatizer = WordNetLemmatizer()
Lemmatizer

<WordNetLemmatizer>

In [39]:
Lemmatizer.lemmatize('fairly', pos='r')

'fairly'

In [ ]:
def lemmatize(words):
    lemmatized_words = []
    for word in words:
        lemmatized_words.append(Lemmatizer.lemmatize(word))
    return lemmatized_words
df['lemma_word'] = df['without_stopwords'].apply(lemmatize)
df[['without_stopwords', 'stemmed_tokens', 'lemma_word']]



,without_stopwords,stemmed_tokens,lemma_word
0,"[would, recommend, item, review]","[would, recommend, item, review]","[would, recommend, item, review]"
1,"[five, stars, completely, satisfied, review]","[five, star, complet, satisfi, review]","[five, star, completely, satisfied, review]"
2,"[customer, service, unhelpful, review]","[custom, servic, unhelp, review]","[customer, service, unhelpful, review]"
3,"[good, value, money, review]","[good, valu, money, review]","[good, value, money, review]"
4,"[excellent, quality, fast, delivery, review]","[excel, qualiti, fast, deliveri, review]","[excellent, quality, fast, delivery, review]"
...,...,...,...
495,"[good, value, money, review]","[good, valu, money, review]","[good, value, money, review]"
496,"[bad, experience, overall, review]","[bad, experi, overal, review]","[bad, experience, overall, review]"
497,"[disappointed, purchase, review]","[disappoint, purchas, review]","[disappointed, purchase, review]"
498,"[product, stopped, working, quickly, review]","[product, stop, work, quickli, review]","[product, stopped, working, quickly, review]"


In [42]:
df['lemma_text'] = df['lemma_word'].apply(lambda x: ' '.join(x))
df.lemma_text

0                 would recommend item review
1       five star completely satisfied review
2           customer service unhelpful review
3                     good value money review
4      excellent quality fast delivery review
                        ...                  
495                   good value money review
496             bad experience overall review
497              disappointed purchase review
498    product stopped working quickly review
499                    useful easy use review
Name: lemma_text, Length: 500, dtype: object

In [43]:
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(df['lemma_text'])
Y = df['sentiment']

print('TF-IDF Shape: ', X.shape)
print('Target Shape: ', Y.shape)

TF-IDF Shape:  (500, 51)
Target Shape:  (500,)


In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)
print('Train Data: ', X_train.shape, Y_train.shape)
print('Test Data: ', X_test.shape, Y_test.shape)